In [4]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

#Load Dataset

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
data = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/IMDB Dataset.csv')

In [7]:
data.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [8]:
data.shape

(50000, 2)

In [9]:
data.tail()

,review,sentiment
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative
49999,No one expects the Star Trek movies to be high...,negative


In [10]:
data["sentiment"].value_counts()

,count
sentiment,
positive,25000
negative,25000


#Label Encorder

In [11]:
data.replace({"sentiment": {"positive": 1, "negative": 0}}, inplace=True)

In [12]:
data.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,1
1,A wonderful little production. <br /><br />The...,1
2,I thought this was a wonderful way to spend ti...,1
3,Basically there's a family where a little boy ...,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",1


In [13]:
#Data Preprocessing

In [3]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, LSTM
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [14]:
train_data, test_data = train_test_split(data, test_size = 0.2, random_state=42)

In [15]:
train_data.shape

(40000, 2)

In [16]:
test_data.shape

(10000, 2)

In [17]:
tokenizer = Tokenizer(num_words = 5000)
tokenizer.fit_on_texts(train_data["review"])

In [18]:
x_train = pad_sequences(tokenizer.texts_to_sequences(train_data["review"]), maxlen=200)
x_test = pad_sequences(tokenizer.texts_to_sequences(test_data["review"]), maxlen=200)

In [23]:
x_train

array([[1935,    1, 1200, ...,  205,  351, 3856],
       [   3, 1651,  595, ...,   89,  103,    9],
       [   0,    0,    0, ...,    2,  710,   62],
       ...,
       [   0,    0,    0, ..., 1641,    2,  603],
       [   0,    0,    0, ...,  245,  103,  125],
       [   0,    0,    0, ...,   70,   73, 2062]], dtype=int32)

In [19]:
Y_train = train_data["sentiment"]

In [21]:
Y_test = test_data["sentiment"]

In [22]:
Y_train

,sentiment
39087,0
30893,0
45278,1
16398,0
13653,0
...,...
11284,1
44732,1
38158,0
860,1


#Model Building

In [26]:
model= Sequential()
model.add(Embedding(input_dim=5000, output_dim=128, input_length=200))
model.add(LSTM(128, dropout=0.2, recurrent_dropout=0.2))
model.add(Dense(1, activation='sigmoid'))


In [27]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [28]:
model.compile(optimizer = "adam", loss="binary_crossentropy", metrics=["accuracy"])

In [30]:
model.fit(x_train, Y_train, epochs=5, batch_size=64, validation_split = 0.2)

Epoch 1/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 214s 410ms/step - accuracy: 0.7282 - loss: 0.5305 - val_accuracy: 0.8524 - val_loss: 0.3459
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 206s 412ms/step - accuracy: 0.8593 - loss: 0.3463 - val_accuracy: 0.8562 - val_loss: 0.3556
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 206s 413ms/step - accuracy: 0.8731 - loss: 0.3119 - val_accuracy: 0.8714 - val_loss: 0.3247
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 263s 415ms/step - accuracy: 0.8945 - loss: 0.2629 - val_accuracy: 0.8659 - val_loss: 0.3212
Epoch 5/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 211s 421ms/step - accuracy: 0.9174 - loss: 0.2153 - val_accuracy: 0.8783 - val_loss: 0.3324


In [33]:
loss, accuracy = model.evaluate(x_test, Y_test)

313/313 ━━━━━━━━━━━━━━━━━━━━ 38s 119ms/step - accuracy: 0.8796 - loss: 0.3119


In [34]:
print(loss)

0.3117125630378723


In [35]:
print(accuracy)

0.8824999928474426


#Building Predictive system

In [36]:
def predictive_system(review):
    sequences = tokenizer.texts_to_sequences([review])
    padded_sequence = pad_sequences(sequences, maxlen=200)
    prediction = model.predict(padded_sequence)
    sentiment = "positive" if prediction[0][0] > 0.5 else "negative"
    return sentiment


In [37]:
predictive_system("This movie was fantastic and amazing")

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 891ms/step


'positive'

In [38]:
predictive_system("A thrilling adventure with stunning visuals")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 221ms/step


'positive'

In [39]:
predictive_system("A visual masterpiece")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 380ms/step


'positive'

#Model Saving

In [40]:
model.save("model.h5")

import joblib
joblib.dump(tokenizer, "tokenizer.pkl")

['tokenizer.pkl']